In [4]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from model import LinearRegressionModel, LogisticRegressionModel


# 加载 task 任务的 train/valid 数据集，特征取自 layer 层
def load_split(task: str, split: str, layer: int) -> tuple[np.ndarray, np.ndarray]:

    data_dir = Path(os.getcwd()).parent / "main" / "datasets" / task

    features = np.load(data_dir / f"{split}.layer{layer}.npy", allow_pickle=False)
    labels = pd.read_csv(data_dir / f"{split}.csv")
    if task == "world":
        targets = labels[["latitude", "longitude"]].to_numpy(dtype=np.float64)
        # 形状为 (N, 2)
    else:
        targets = labels["label"].to_numpy(dtype=np.float64)
        # 形状为 (N,)
    if len(features) != len(targets):
        raise ValueError("The feature and label files have different row counts")

    return features.astype(np.float64), targets


In [ ]:
"""
world/layer11
数据获取/初步查看情况
"""

task = "world"
split = "train"
layer = 11

data_dir = Path(os.getcwd()).parent / "main" / "datasets" / task

features = np.load(data_dir / f"{split}.layer{layer}.npy", allow_pickle=False)
labels = pd.read_csv(data_dir / f"{split}.csv")
if task == "world":
    targets = labels[["latitude", "longitude"]].to_numpy(dtype=np.float64)
    # 形状为 (N, 2)
else:
    targets = labels["label"].to_numpy(dtype=np.float64)
    # 形状为 (N,)

# 检查行数是否匹配
if len(features) != len(targets):
    raise ValueError("The feature and label files have different row counts")

print("")

In [2]:
"""
world/layer 11
训练Baseline
"""

task = "world"
layer = 11
output_dir = Path(os.getcwd()).parent.parent / "output" / "world"
lr = 0.01
l1 = 0.01
l2 = 0.01
max_iter = 10000
# ================================================

# 加载数据
train_x, train_y = load_split(task, "train", layer)
valid_x, valid_y = load_split(task, "valid", layer)

# 纬度和经度各训练一个一维线性回归模型。
latitude_targets = train_y[:, 0]
longitude_targets = train_y[:, 1]
latitude_model = LinearRegressionModel(
    n_features=train_x.shape[1],
    l1=l1,
    l2=l2,
    lr=lr,
    max_iter=max_iter,
)
longitude_model = LinearRegressionModel(
    n_features=train_x.shape[1],
    l1=l1,
    l2=l2,
    lr=lr,
    max_iter=max_iter,
)

# 训练
latitude_model.fit(train_x, latitude_targets)
longitude_model.fit(train_x, longitude_targets)

# 测试
latitude_predictions = latitude_model.predict(valid_x)
longitude_predictions = longitude_model.predict(valid_x)
prediction = np.column_stack((latitude_predictions, longitude_predictions))
## 防止错误
if prediction.shape != valid_y.shape:
    raise ValueError(f"Expected predictions with shape {valid_y.shape}")

# 输出结果

# 1 打印验证集上的MSE
print(f"Validation MSE: {np.mean((prediction - valid_y) ** 2):.4f}")

# 2 参数
weights = np.stack((latitude_model.weights, longitude_model.weights))
bias = np.array((latitude_model.bias, longitude_model.bias))
## 创造输出目录（如果不存在的话）
output_dir.mkdir(parents=True, exist_ok=True)
output = output_dir / f"{task}_layer{layer}.npz"
## 输出文件的名称被 output 参数指定，文件格式为 .npz
np.savez(
    output,
    weights=np.asarray(weights, dtype=np.float64),
    bias=np.asarray(bias, dtype=np.float64),
)
print(f"Saved {output} (layer {layer})")



/home/serelive/Workspace/Courses/AI3002/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:132: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/home/serelive/Workspace/Courses/AI3002/assignments/labs/lab1/main/model.py:46: RuntimeWarning: invalid value encountered in matmul
  gradient_w = 2 / features.shape[0] * features.T @ residual + self.l1 * np.sign(weights) + 2 * self.l2 * weights
/home/serelive/Workspace/Courses/AI3002/assignments/labs/lab1/main/model.py:31: RuntimeWarning: invalid value encountered in scalar subtract
  bias = bias - self.lr * gradient[1]


Validation MSE: nan
Saved /home/serelive/Workspace/Courses/AI3002/assignments/labs/output/world/world_layer11.npz (layer 11)


In [2]:
"""
world/layer 11
训练Baseline
"""

task = "world"
layer = 11
output_dir = Path(os.getcwd()).parent.parent / "output" / "world"
lr = 0.01
l1 = 0.01
l2 = 0.01
max_iter = 10000
# ================================================

# 加载数据
train_x, train_y = load_split(task, "train", layer)
valid_x, valid_y = load_split(task, "valid", layer)

if task == "world":
    # 纬度和经度各训练一个一维线性回归模型。
    latitude_targets = train_y[:, 0]
    longitude_targets = train_y[:, 1]
    latitude_model = LinearRegressionModel(
        n_features=train_x.shape[1],
        l1=l1,
        l2=l2,
        lr=lr,
        max_iter=max_iter,
    )
    longitude_model = LinearRegressionModel(
        n_features=train_x.shape[1],
        l1=l1,
        l2=l2,
        lr=lr,
        max_iter=max_iter,
    )
    latitude_model.fit(train_x, latitude_targets)
    longitude_model.fit(train_x, longitude_targets)
    latitude_predictions = latitude_model.predict(valid_x)
    longitude_predictions = longitude_model.predict(valid_x)
    prediction = np.column_stack((latitude_predictions, longitude_predictions))
    weights = np.stack((latitude_model.weights, longitude_model.weights))
    bias = np.array((latitude_model.bias, longitude_model.bias))
else:
    model = LogisticRegressionModel(train_x.shape[1])
    model.fit(train_x, train_y)
    prediction = model.predict(valid_x)
    weights = model.weights
    bias = model.bias

if prediction.shape != valid_y.shape:
    raise ValueError(f"Expected predictions with shape {valid_y.shape}")

if task == "world":
    print(f"Validation MSE: {np.mean((prediction - valid_y) ** 2):.4f}")
else:
    print(f"Validation accuracy: {np.mean((prediction >= 0.5) == valid_y):.4f}")

# 创造输出目录（如果不存在的话）
output_dir.mkdir(parents=True, exist_ok=True)
output = output_dir / f"{task}_layer{layer}.npz"

# 输出文件的名称被 output 参数指定，文件格式为 .npz
np.savez(
    output,
    weights=np.asarray(weights, dtype=np.float64),
    bias=np.asarray(bias, dtype=np.float64),
)
print(f"Saved {output} (layer {layer})")

/home/serelive/Workspace/Courses/AI3002/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:132: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/home/serelive/Workspace/Courses/AI3002/assignments/labs/lab1/main/model.py:46: RuntimeWarning: invalid value encountered in matmul
  gradient_w = 2 / features.shape[0] * features.T @ residual + self.l1 * np.sign(weights) + 2 * self.l2 * weights
/home/serelive/Workspace/Courses/AI3002/assignments/labs/lab1/main/model.py:31: RuntimeWarning: invalid value encountered in scalar subtract
  bias = bias - self.lr * gradient[1]


Validation MSE: nan
Saved /home/serelive/Workspace/Courses/AI3002/assignments/labs/output/world/world_layer11.npz (layer 11)
